In [6]:
import pandas as pd
import datetime
dfs = []

for day in range(1, 31):      # 1 → 30
    filename = f"/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/anonymized_raw_2025-04-{day:02d}.csv"
    df = pd.read_csv(filename)
    dfs.append(df)

raw_all = pd.concat(dfs, ignore_index=True)



/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_25497/1909453941.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)
/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_25497/1909453941.py:7: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


In [7]:

raw_all['datetime'] = pd.to_datetime(raw_all['datetime'])
start = pd.to_datetime('05:00').time()
end = pd.to_datetime('22:15').time()

raw_all = raw_all[raw_all['datetime'].dt.time.between(start, end)]

In [8]:
raw_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55186579 entries, 242657 to 63825230
Data columns (total 8 columns):
 #   Column              Dtype         
---  ------              -----         
 0   datetime            datetime64[ns]
 1   lng                 float64       
 2   lat                 float64       
 3   speed               float64       
 4   door_up             bool          
 5   door_down           bool          
 6   anonymized_vehicle  object        
 7   anonymized_driver   object        
dtypes: bool(2), datetime64[ns](1), float64(3), object(2)
memory usage: 3.0+ GB


In [9]:
raw_all.drop_duplicates(inplace=True)

In [10]:
raw_all = raw_all.sort_values(['anonymized_vehicle','datetime' ], ascending=[True,True])


In [11]:
raw_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54459607 entries, 244703 to 63467446
Data columns (total 8 columns):
 #   Column              Dtype         
---  ------              -----         
 0   datetime            datetime64[ns]
 1   lng                 float64       
 2   lat                 float64       
 3   speed               float64       
 4   door_up             bool          
 5   door_down           bool          
 6   anonymized_vehicle  object        
 7   anonymized_driver   object        
dtypes: bool(2), datetime64[ns](1), float64(3), object(2)
memory usage: 2.9+ GB


In [13]:
raw_all.to_csv('raw_all.csv', index=False, encoding='utf-8')

In [ ]:
rev_stop_merged = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/rev_stop_merged.csv')
stop_by_var = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Stop_by_vars_merged.csv')

HCM_bus_routes = pd.concat([rev_stop_merged,stop_by_var])
HCM_bus_routes.drop_duplicates('StopId',inplace=True)


In [ ]:
HCM_bus_routes[HCM_bus_routes.duplicated(keep=False)]

,StopId,Code,Name,StopType,Zone,Ward,AddressNo,Street,SupportDisability,Status,Lng,Lat,Search,Routes,Column1


In [ ]:
HCM_bus_routes.to_csv('HCM_bus_routes.csv', index=False, encoding='utf-8')

<class 'pandas.core.frame.DataFrame'>
Index: 2152 entries, 0 to 1172
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   StopId             2152 non-null   int64  
 1   Code               2152 non-null   object 
 2   Name               2152 non-null   object 
 3   StopType           2152 non-null   object 
 4   Zone               2152 non-null   object 
 5   Ward               898 non-null    object 
 6   AddressNo          2152 non-null   object 
 7   Street             2152 non-null   object 
 8   SupportDisability  210 non-null    object 
 9   Status             2152 non-null   object 
 10  Lng                2152 non-null   float64
 11  Lat                2152 non-null   float64
 12  Search             2152 non-null   object 
 13  Routes             2152 non-null   object 
 14  Column1            28 non-null     object 
dtypes: float64(2), int64(1), object(12)
memory usage: 269.0+ KB


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point



df_gps = df_gps[df_gps['datetime'].dt.time.between(start, end)]
df_station = pd.read_csv("/Users/voquangkhai/Desktop/Bus_route_data/HCMC_bus_routes/1/rev_stops_by_var.csv")  # thay file thật của bạn

# --- 2. Convert sang GeoDataFrame ---
gdf_gps = gpd.GeoDataFrame(
    df_gps,
    geometry=gpd.points_from_xy(df_gps['lng'], df_gps['lat']),
    crs="EPSG:4326"
)

gdf_station = gpd.GeoDataFrame(
    df_station,
    geometry=gpd.points_from_xy(df_station['Lng'], df_station['Lat']),
    crs="EPSG:4326"
)

# --- 3. Chuyển sang hệ toạ độ mét ---
gdf_gps = gdf_gps.to_crs(epsg=3857)
gdf_station = gdf_station.to_crs(epsg=3857)

# --- 4. Join theo khoảng cách (ví dụ 30 mét) ---
distance_threshold = 10  # mét

gdf_joined = gpd.sjoin_nearest(
    gdf_gps,
    gdf_station,
    how="inner",
    max_distance=distance_threshold
)

# Kết quả: thêm cột station_id, station_name, distance_m
display(gdf_joined)

In [2]:
import pandas as pd
from ydata_profiling import ProfileReport
gdf_joined = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/gdf_joined.csv')
ProfileReport(gdf_joined, title="Data Report").to_file("report.html")

/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_3864/3086365219.py:3: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  gdf_joined = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/gdf_joined.csv')


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 25/25 [00:39<00:00,  1.58s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:


2.  Tiền xử lý: Chuyển đổi cột ngày tháng sang định dạng datetime
Giả sử cột chứa thông tin ngày là 'Ngay_Thang'. Thay thế nếu tên cột khác.
df['Ngay_Thang'] = pd.to_datetime(df['Ngay_Thang'])

3.  Định nghĩa Giai đoạn 1 (Baseline)
ngay_bat_dau = '2024-04-01'
ngay_ket_thuc = '2024-04-26'

4.  Lọc dữ liệu cho Giai đoạn 1
df_giai_doan_1 = df[
    (df['Ngay_Thang'] >= ngay_bat_dau) & 
    (df['Ngay_Thang'] <= ngay_ket_thuc)
]


df_hoan_thanh_giai_doan_1 = df_giai_doan_1[
    df_giai_doan_1['Trang_Thai'].str.contains('Hoàn thành', case=False, na=False)
]


tong_so_chuyen_giai_doan_1 = df_hoan_thanh_giai_doan_1.shape[0]


print("\n--- KẾT QUẢ PHÂN TÍCH GIAI ĐOẠN 1 (BASELINE) ---")
print(f"Giai đoạn phân tích: Từ {ngay_bat_dau} đến {ngay_ket_thuc}")
print(f"Tổng số chuyến xe HOÀN THÀNH trong Giai đoạn 1 là: {tong_so_chuyen_giai_doan_1} chuyến.")


tong_tat_ca_chuyen_giai_doan_1 = df_giai_doan_1.shape[0]
print(f"Tổng số chuyến xe (bao gồm cả bị hủy/rút ngắn): {tong_tat_ca_chuyen_giai_doan_1} chuyến.")


In [ ]:
import matplotlib as plt
import seaborn as sns
df = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/gdf_joined.csv')


/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_3864/2303860118.py:3: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/nguyentran0703/Downloads/BDC HACKATHON/Bus_route_data/raw_GPS/gdf_joined.csv')


In [9]:
df['Status']

0                Không có
1          Đang khai thác
2          Đang khai thác
3          Đang khai thác
4          Đang khai thác
                ...      
2653513    Đang khai thác
2653514    Đang khai thác
2653515    Đang khai thác
2653516    Đang khai thác
2653517    Đang khai thác
Name: Status, Length: 2653518, dtype: object

In [10]:
df.head()

,datetime,lng,lat,speed,door_up,door_down,anonymized_vehicle,anonymized_driver,geometry,index_right,...,Ward,AddressNo,Street,SupportDisability,Status,Lng,Lat,Search,Routes,Column1
0,2025-04-01 05:01:11,106.859550,10.808240,6.0,False,False,001d86aa2e,NaN,POINT (11895550.692398356 1210367.668029209),752,...,NaN,Đầu bến tuyến 88,Long Phước,NaN,Không có,106.859556,10.808194,CLP Dbt88 LP,76,NaN
1,2025-04-01 05:38:36,106.779420,10.789523,12.0,False,False,001d86aa2e,NaN,POINT (11886630.661601093 1208246.5755954601),1476,...,NaN,672-674,Nguyễn Duy Trinh,NaN,Đang khai thác,106.779427,10.789444,DXH 672-674 NDT,"29, 88",NaN
2,2025-04-01 05:55:38,106.745163,10.791047,21.0,False,False,001d86aa2e,NaN,POINT (11882817.226911481 1208419.20453725),834,...,NaN,Đối diện siêu thị điện máy Chợ Lớn,Lương Định Của,NaN,Đang khai thác,106.745251,10.791052,StCL DdstdmCL LDC,"157V, 43, 88",NaN
3,2025-04-01 06:14:43,106.701793,10.771048,19.0,False,False,001d86aa2e,NaN,POINT (11877989.300595777 1206152.9998702733),231,...,Phường Bến Nghé,Trường Cao Thắng,Hàm Nghi,NaN,Đang khai thác,106.701853,10.771043,TCT TCT HN,"01, 03, 102, 19, 20, 34, 38, 39, 45, 56, 75, 8...",NaN
4,2025-04-01 06:16:43,106.699597,10.771088,8.0,False,False,001d86aa2e,NaN,POINT (11877744.768781003 1206157.5325070592),845,...,Phường Bến Thành,Hàm Nghi 8,Hàm Nghi,Có,Đang khai thác,106.699514,10.771117,TTctdHN HN8 HN,"19, 56, 88",NaN


In [23]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

# Thiết lập style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial', 'sans-serif']

# ----------------------------------------------------------------------
# 1. Tiền xử lý Dữ liệu (Dựa trên biến df có sẵn từ bước trước)
# ----------------------------------------------------------------------

# Chuyển đổi cột datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Tạo cột Giờ để vẽ biểu đồ (thay cho cột Gio_Trong_Ngay giả định cũ)
df['hour'] = df['datetime'].dt.hour

# Lọc dữ liệu: Chọn các trạm/chuyến "Đang khai thác" (Thay vì 'Hoàn thành')
# Lưu ý: Nếu muốn lấy tất cả dữ liệu GPS bất kể trạng thái, bạn có thể bỏ dòng lọc này.
df_clean = df[df['Status'].str.contains('Đang khai thác', case=False, na=False)].copy()

# 2. Định nghĩa Giai đoạn (Cập nhật năm 2025 theo dữ liệu mẫu)
ngay_bat_dau_g1 = '2025-04-01'
ngay_ket_thuc_g1 = '2025-04-26'
ngay_bat_dau_g2 = '2025-04-27'
ngay_ket_thuc_g2 = '2025-04-30'

# Gán nhãn giai đoạn
df_clean['Giai_Doan'] = np.where(
    (df_clean['datetime'] >= ngay_bat_dau_g2) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2), 
    'Giai đoạn Sự kiện (27-30/4)', 
    'Giai đoạn trước sự kiện  (1-26/4)'
)

# Chỉ giữ lại các dòng thuộc 1 trong 2 giai đoạn này (loại bỏ ngày tháng khác nếu có)
df_final = df_clean[
    (df_clean['datetime'] >= ngay_bat_dau_g1) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2)
]

# ----------------------------------------------------------------------
# PHẦN A: BIỂU ĐỒ LINE CHART SO SÁNH TỐC ĐỘ (speed) THEO GIỜ (hour)
# ----------------------------------------------------------------------

# Gom nhóm dữ liệu: Tính tốc độ trung bình theo Giai đoạn và Giờ
# 'speed' là tên cột trong dữ liệu của bạn
df_toc_do_theo_gio = df_final.groupby(['Giai_Doan', 'hour'])['speed'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_toc_do_theo_gio, 
    x='hour', 
    y='speed', 
    hue='Giai_Doan', 
    marker='o',
    linewidth=2.5,
    palette={'Giai đoạn trước sự kiện  (1-26/4)': '#1f77b4', 'Giai đoạn Sự kiện (27-30/4)': '#d62728'}
)

# Trang trí biểu đồ
plt.title('So Sánh Tốc Độ Di Chuyển Trung Bình Theo Giờ: Ngày Thường vs Lễ', fontsize=15, fontweight='bold')
plt.xlabel('Giờ Trong Ngày (0h - 23h)', fontsize=12)
plt.ylabel('Tốc độ Trung bình (km/h)', fontsize=12)

# Thiết lập trục X hiển thị đủ 24h
plt.xticks(np.arange(0, 24, 1))

plt.legend(title='Giai đoạn', loc='lower center') # Đưa chú thích xuống dưới để đỡ che biểu đồ
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()

/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_3864/246396370.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# ... (Các phần code xử lý dữ liệu giữ nguyên) ...

# ----------------------------------------------------------------------
# PHẦN A: BIỂU ĐỒ LINE CHART SO SÁNH TỐC ĐỘ (speed) THEO GIỜ (hour)
# ----------------------------------------------------------------------

# Gom nhóm dữ liệu: Tính tốc độ trung bình theo Giai đoạn và Giờ
df_toc_do_theo_gio = df_final.groupby(['Giai_Doan', 'hour'])['speed'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_toc_do_theo_gio, 
    x='hour', 
    y='speed', 
    hue='Giai_Doan', 
    marker='o',
    linewidth=2.5,
    palette={'Giai đoạn trước sự kiện  (1-26/4)': '#1f77b4', 'Giai đoạn Sự kiện (27-30/4)': '#d62728'}
)

# Trang trí biểu đồ
plt.title('So Sánh Tốc Độ Di Chuyển Trung Bình Theo Giờ: Ngày Thường vs Lễ', fontsize=15, fontweight='bold')
plt.xlabel('Giờ Trong Ngày (0h - 23h)', fontsize=12)
plt.ylabel('Tốc độ Trung bình (km/h)', fontsize=12)

# Thiết lập trục X hiển thị đủ 24h
plt.xticks(np.arange(0, 24, 1))

plt.legend(title='Giai đoạn', loc='lower center')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# --- THAY ĐỔI Ở ĐÂY ---
# Thay vì plt.show(), ta dùng plt.savefig
file_name = 'bieu_do_toc_do_theo_gio.png'
plt.savefig(file_name, dpi=300) 
print(f"Đã lưu biểu đồ thành công vào file: {file_name}")
# Bạn hãy mở thư mục chứa file code để xem ảnh vừa tạo

Đã lưu biểu đồ thành công vào file: bieu_do_toc_do_theo_gio.png


In [25]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

# Thiết lập style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial', 'sans-serif']

# ----------------------------------------------------------------------
# 1. Chuẩn bị dữ liệu (Tiếp nối từ các bước trước)
# ----------------------------------------------------------------------
# Đảm bảo cột datetime đã đúng định dạng
df['datetime'] = pd.to_datetime(df['datetime'])

# Lọc dữ liệu "Đang khai thác"
df_clean = df[df['Status'].str.contains('Đang khai thác', case=False, na=False)].copy()

# Định nghĩa mốc thời gian (Năm 2025)
ngay_bat_dau_g1 = '2025-04-01'
ngay_ket_thuc_g1 = '2025-04-26'
ngay_bat_dau_g2 = '2025-04-27'
ngay_ket_thuc_g2 = '2025-04-30'

# Gán nhãn Giai đoạn
df_clean['Giai_Doan'] = np.where(
    (df_clean['datetime'] >= ngay_bat_dau_g2) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2), 
    'Giai đoạn Sự kiện (27-30/4)', 
    'Giai đoạn trước sự kiện  (1-26/4)'
)

# Chỉ lấy dữ liệu trong khoảng thời gian quan tâm
df_final = df_clean[
    (df_clean['datetime'] >= ngay_bat_dau_g1) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2)
]

# ----------------------------------------------------------------------
# PHẦN B: BIỂU ĐỒ 2 - BAR CHART SO SÁNH HOẠT ĐỘNG THEO TUYẾN (Routes)
# ----------------------------------------------------------------------

# BƯỚC 1: Tính tổng số XE (Unique Vehicles) theo từng giai đoạn và tuyến
# Thay vì đếm dòng (count), ta đếm số lượng xe duy nhất (nunique) để đại diện cho tần suất
df_chuyen_theo_tuyen = df_final.groupby(['Giai_Doan', 'Routes'])['anonymized_vehicle'].nunique().reset_index()
df_chuyen_theo_tuyen.columns = ['Giai_Doan', 'Routes', 'Tong_So_Xe_Hoat_Dong']

# BƯỚC 2: Chuẩn hóa thời gian (Tính trung bình theo ngày)
# Cập nhật ngày bắt đầu thành 2025-04-01 để tính đúng số ngày chia
so_ngay_g1 = (pd.to_datetime(ngay_ket_thuc_g1) - pd.to_datetime('2025-04-01')).days + 1
so_ngay_g2 = (pd.to_datetime(ngay_ket_thuc_g2) - pd.to_datetime(ngay_bat_dau_g2)).days + 1

df_chuyen_theo_tuyen['So_Ngay'] = df_chuyen_theo_tuyen['Giai_Doan'].apply(
    lambda x: so_ngay_g2 if 'Sự kiện' in x else so_ngay_g1
)

# Tính chỉ số: Số xe trung bình hoạt động/ngày
df_chuyen_theo_tuyen['Xe_TB_Ngay'] = df_chuyen_theo_tuyen['Tong_So_Xe_Hoat_Dong'] / df_chuyen_theo_tuyen['So_Ngay']

# Lọc bớt các tuyến ít dữ liệu (Optional) để biểu đồ đỡ rối nếu có quá nhiều tuyến
# df_chuyen_theo_tuyen = df_chuyen_theo_tuyen[df_chuyen_theo_tuyen['Tong_So_Xe_Hoat_Dong'] > 5]

# BƯỚC 3: Vẽ biểu đồ
plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_chuyen_theo_tuyen, 
    x='Routes', 
    y='Xe_TB_Ngay', 
    hue='Giai_Doan', 
    palette={'Giai đoạn trước sự kiện  (1-26/4)': '#2CA02C', 'Giai đoạn Sự kiện (27-30/4)': '#FF7F0E'}
)

plt.title('So Sánh Số Lượng Xe Hoạt Động Trung Bình/Ngày Theo Tuyến', fontsize=15, fontweight='bold')
plt.xlabel('Tuyến (Routes)', fontsize=12)
plt.ylabel('Số Xe Trung Bình/Ngày', fontsize=12)
plt.xticks(rotation=45) # Xoay tên tuyến nếu tên quá dài
plt.legend(title='Giai đoạn', loc='upper right')
plt.tight_layout()

# Lưu file thay vì show() để tránh lỗi
file_name = 'bieu_do_so_sanh_tuyen.png'
plt.savefig(file_name, dpi=300)
print(f"Đã lưu biểu đồ vào file: {file_name}")

/var/folders/nx/tm25jbz54jjdb9mb8c28s0m00000gn/T/ipykernel_3864/2734621039.py:78: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Đã lưu biểu đồ vào file: bieu_do_so_sanh_tuyen.png


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ----------------------------------------------------------------------
# 1. Chuẩn bị dữ liệu
# ----------------------------------------------------------------------
# Đảm bảo cột datetime đúng định dạng
df['datetime'] = pd.to_datetime(df['datetime'])

# Lọc dữ liệu "Đang khai thác"
df_clean = df[df['Status'].str.contains('Đang khai thác', case=False, na=False)].copy()

# Lọc lấy dữ liệu cả tháng 4/2025 (để thấy xu hướng cả tháng)
ngay_bat_dau_plot = '2025-04-01'
ngay_ket_thuc_plot = '2025-04-30'

df_final = df_clean[
    (df_clean['datetime'] >= ngay_bat_dau_plot) & 
    (df_clean['datetime'] <= ngay_ket_thuc_plot)
]

# ----------------------------------------------------------------------
# PHẦN C: BIỂU ĐỒ 3 - LINE CHART TỐC ĐỘ TRUNG BÌNH THEO NGÀY
# (Thay thế cho Độ trễ vì dữ liệu GPS không có cột Delay)
# ----------------------------------------------------------------------

# Tính tốc độ trung bình theo ngày
# dt.date giúp gom nhóm theo ngày (bỏ qua giờ phút)
df_toc_do_ngay = df_final.groupby(df_final['datetime'].dt.date)['speed'].mean().reset_index()
df_toc_do_ngay.columns = ['Ngay', 'Toc_Do_TB']
df_toc_do_ngay['Ngay'] = pd.to_datetime(df_toc_do_ngay['Ngay']) # Chuyển lại datetime để vẽ trục X

plt.figure(figsize=(12, 6))

# Vẽ Line Chart
# Dùng màu đỏ (#d62728) để thể hiện sự cảnh báo nếu tốc độ tụt giảm
plt.plot(df_toc_do_ngay['Ngay'], df_toc_do_ngay['Toc_Do_TB'], 
         marker='o', linestyle='-', color='#d62728', linewidth=2, label='Tốc độ TB')

# Đánh dấu vùng sự kiện (27/4 - 30/4) - Lưu ý cập nhật năm 2025
ngay_su_kien_start = pd.to_datetime('2025-04-27')
ngay_su_kien_end = pd.to_datetime('2025-04-30')

# Vẽ vùng màu nền để làm nổi bật giai đoạn sự kiện
plt.axvspan(ngay_su_kien_start, ngay_su_kien_end, color='orange', alpha=0.2, label='Giai đoạn Sự kiện (27-30/4)')

# Thiết lập tiêu đề và nhãn
plt.title('Biến Động Tốc Độ Di Chuyển Trung Bình Hàng Ngày (Tháng 4/2025)', fontsize=15, fontweight='bold')
plt.xlabel('Ngày', fontsize=12)
plt.ylabel('Tốc độ Trung bình (km/h)', fontsize=12)

# Định dạng trục X: Hiển thị ngày/tháng
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=2)) # Hiển thị cách ngày cho đỡ rối
plt.xticks(rotation=45)

plt.grid(True, linestyle='--', alpha=0.7, axis='y')
plt.legend()
plt.tight_layout() 

# Lưu file
file_name = 'bieu_do_toc_do_theo_ngay.png'
plt.savefig(file_name, dpi=300)
print(f"\nĐã tạo và lưu biểu đồ vào file: {file_name}")


Đã tạo và lưu biểu đồ vào file: bieu_do_toc_do_theo_ngay.png


In [ ]:
gdf_joined.columns


Index(['datetime', 'lng', 'lat', 'speed', 'door_up', 'door_down',
       'anonymized_vehicle', 'anonymized_driver', 'geometry', 'index_right',
       'StopId', 'Code', 'Name', 'StopType', 'Zone', 'Ward', 'AddressNo',
       'Street', 'SupportDisability', 'Status', 'Lng', 'Lat', 'Search',
       'Routes', 'Column1'],
      dtype='object')

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CHUẨN BỊ DỮ LIỆU LẠI TỪ ĐẦU (ĐỂ ĐẢM BẢO KHÔNG BỊ LỖI THIẾU CỘT) ---
# Đảm bảo cột datetime đúng định dạng
df['datetime'] = pd.to_datetime(df['datetime'])

# Lọc dữ liệu "Đang khai thác"
df_clean = df[df['Status'].str.contains('Đang khai thác', case=False, na=False)].copy()

# Định nghĩa ngày
ngay_bat_dau_g1 = '2025-04-01'
ngay_ket_thuc_g1 = '2025-04-26'
ngay_bat_dau_g2 = '2025-04-27'
ngay_ket_thuc_g2 = '2025-04-30'

# --- QUAN TRỌNG: TẠO LẠI CỘT 'Giai_Doan' TẠI ĐÂY ---
df_clean['Giai_Doan'] = np.where(
    (df_clean['datetime'] >= ngay_bat_dau_g2) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2), 
    'Giai đoạn Sự kiện (27-30/4)', 
    'Giai đoạn trước sự kiện  (1-26/4)'
)

# Chỉ lấy dữ liệu trong khoảng thời gian quan tâm (Tháng 4/2025)
df_final = df_clean[
    (df_clean['datetime'] >= ngay_bat_dau_g1) & 
    (df_clean['datetime'] <= ngay_ket_thuc_g2)
].copy()

# --- 2. XỬ LÝ TÁCH TUYẾN (Fix lỗi biểu đồ trắng) ---
# Tách cột Routes (ví dụ "29, 88" -> thành 2 dòng riêng biệt)
# Lưu ý: Kiểm tra xem dữ liệu của bạn ngăn cách bằng ", " (phẩy + cách) hay chỉ ","
df_exploded = df_final.assign(Routes=df_final['Routes'].str.split(', ')).explode('Routes')

# Loại bỏ các dòng rỗng hoặc NaN
df_exploded = df_exploded[df_exploded['Routes'].notna()]
df_exploded = df_exploded[df_exploded['Routes'].str.strip() != '']

# --- 3. TÍNH TOÁN SỐ LIỆU ---
# Bước 3.1: Đếm số xe unique theo Giai đoạn và Tuyến
df_chuyen_theo_tuyen = df_exploded.groupby(['Giai_Doan', 'Routes'])['anonymized_vehicle'].nunique().reset_index()
df_chuyen_theo_tuyen.columns = ['Giai_Doan', 'Routes', 'Tong_So_Xe_Hoat_Dong']

# Bước 3.2: Tính số ngày để chuẩn hóa
so_ngay_g1 = (pd.to_datetime(ngay_ket_thuc_g1) - pd.to_datetime(ngay_bat_dau_g1)).days + 1
so_ngay_g2 = (pd.to_datetime(ngay_ket_thuc_g2) - pd.to_datetime(ngay_bat_dau_g2)).days + 1

df_chuyen_theo_tuyen['So_Ngay'] = df_chuyen_theo_tuyen['Giai_Doan'].apply(
    lambda x: so_ngay_g2 if 'Sự kiện' in x else so_ngay_g1
)

# Tính số xe trung bình/ngày
df_chuyen_theo_tuyen['Xe_TB_Ngay'] = df_chuyen_theo_tuyen['Tong_So_Xe_Hoat_Dong'] / df_chuyen_theo_tuyen['So_Ngay']

# --- 4. LỌC DỮ LIỆU ĐỂ VẼ (TOP 20 TUYẾN) ---
# Lấy danh sách Top 20 tuyến hoạt động mạnh nhất ở Baseline để vẽ cho gọn
if not df_chuyen_theo_tuyen.empty:
    top_routes = df_chuyen_theo_tuyen[
        df_chuyen_theo_tuyen['Giai_Doan'] == 'Giai đoạn trước sự kiện  (1-26/4)'
    ].nlargest(20, 'Tong_So_Xe_Hoat_Dong')['Routes']
    
    df_plot = df_chuyen_theo_tuyen[df_chuyen_theo_tuyen['Routes'].isin(top_routes)].copy()
    
    # Sắp xếp dữ liệu
    df_plot.sort_values(by=['Giai_Doan', 'Xe_TB_Ngay'], ascending=[True, False], inplace=True)

    # --- 5. VẼ BIỂU ĐỒ ---
    plt.figure(figsize=(14, 8))
    sns.barplot(
        data=df_plot,
        x='Routes',
        y='Xe_TB_Ngay',
        hue='Giai_Doan',
        palette={'Giai đoạn trước sự kiện  (1-26/4)': '#2CA02C', 'Giai đoạn Sự kiện (27-30/4)': '#FF7F0E'}
    )

    plt.title('So Sánh Số Lượng Xe Hoạt Động Trung Bình/Ngày (Top 20 Tuyến)', fontsize=16, fontweight='bold')
    plt.xlabel('Mã Tuyến', fontsize=12)
    plt.ylabel('Số Xe Trung Bình/Ngày', fontsize=12)
    plt.xticks(rotation=90) # Xoay tên tuyến thẳng đứng
    plt.legend(title='Giai đoạn')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # Lưu file
    plt.savefig('bieu_do_b_so_sanh_tuyen_final.png', dpi=300)
    print("Đã vẽ và lưu biểu đồ thành công: bieu_do_b_so_sanh_tuyen_final.png")
else:
    print("Cảnh báo: Không có dữ liệu sau khi xử lý. Hãy kiểm tra lại tên cột Routes hoặc bộ lọc.")

Đã vẽ và lưu biểu đồ thành công: bieu_do_b_so_sanh_tuyen_final.png
